# Extração SIA/SUS — Produção Ambulatorial para o projeto Conecta Saúde

Este notebook documenta a fonte **SIA/SUS — Sistema de Informações Ambulatoriais do SUS**.

O objetivo é mostrar:

1. **quais dados serão extraídos**;
2. **qual endpoint público será usado**;
3. **como realizar a extração com Python**;
4. **quais campos são úteis para o projeto**;
5. **quais perguntas de negócio podem ser respondidas com esses dados**.

O SIA/SUS será usado como base complementar para o **Tema 4 — Gestão de Espera e Acesso a Especialidades**, principalmente como proxy de demanda ambulatorial e especializada.


## 1. Papel do SIA/SUS no projeto

Dentro do projeto, o SIA/SUS será usado para analisar a **produção ambulatorial especializada**.

A base será usada para analisar:

- consultas e procedimentos ambulatoriais;
- exames especializados;
- quantidade apresentada e aprovada;
- valor aprovado;
- estabelecimento executante, identificado por CNES;
- município de atendimento;
- município de residência do paciente, quando disponível;
- procedimento realizado;
- CBO/profissional associado ao atendimento;
- CID, quando aplicável;
- perfil de atendimento por idade e sexo.

Na arquitetura do projeto, o SIA/SUS será cruzado com:

- **SIGTAP**, para traduzir e classificar códigos de procedimentos;
- **CNES**, para identificar estabelecimento, tipo de unidade e serviços disponíveis;
- **IBGE**, para calcular procedimentos por 10 mil habitantes;
- **SIH/SUS**, para comparar pressão ambulatorial e hospitalar por região.


## 2. Endpoint público de conexão

A estrutura dos arquivos de produção ambulatorial do SIA/SUS segue o padrão:

```text
https://ftp.datasus.gov.br/dissemin/publicos/SIASUS/200801_/Dados/PA{UF}{AAMM}.dbc
```

Exemplo para São Paulo, janeiro de 2024:

```text
https://ftp.datasus.gov.br/dissemin/publicos/SIASUS/200801_/Dados/PASP2401.dbc
```

Onde:

| Parte | Significado |
|---|---|
| `PA` | Arquivo de produção ambulatorial |
| `SP` | Unidade Federativa |
| `24` | Ano com dois dígitos: 2024 |
| `01` | Mês: janeiro |
| `.dbc` | Formato comprimido usado pelo DATASUS |

Para o MVP, comece com poucos meses e uma UF. Depois, amplie o recorte.


In [ ]:
# Instalação das dependências
# Execute esta célula no Colab/Jupyter caso os pacotes ainda não estejam instalados.

%pip install -q pandas pyarrow requests pysus


## 3. Parâmetros iniciais da extração


In [ ]:
from pathlib import Path
from typing import Iterable
import pandas as pd
import requests

UF = "SP"
ANO = 2024
MESES = [1]
GRUPO = "PA"  # PA = Produção Ambulatorial

BASE_URL_SIA_PA = "https://ftp.datasus.gov.br/dissemin/publicos/SIASUS/200801_/Dados/"
DIRETORIO_RAW = Path("dados/raw/sia_pa")
DIRETORIO_TRATADO = Path("dados/tratado/sia_pa")

DIRETORIO_RAW.mkdir(parents=True, exist_ok=True)
DIRETORIO_TRATADO.mkdir(parents=True, exist_ok=True)


## 4. Geração do link do arquivo DBC


In [ ]:
def montar_url_sia_pa(uf: str, ano: int, mes: int) -> str:
    uf = uf.upper().strip()
    aa = str(ano)[-2:]
    mm = str(mes).zfill(2)
    arquivo = f"PA{uf}{aa}{mm}.dbc"
    return f"{BASE_URL_SIA_PA}{arquivo}"

for mes in MESES:
    print(montar_url_sia_pa(UF, ANO, mes))


## 5. Verificação de disponibilidade


In [ ]:
def verificar_disponibilidade(url: str, timeout: int = 30) -> dict:
    try:
        resposta = requests.head(url, timeout=timeout, allow_redirects=True)
        return {
            "url": url,
            "status_code": resposta.status_code,
            "disponivel": resposta.status_code == 200,
            "content_length_bytes": resposta.headers.get("Content-Length"),
            "content_type": resposta.headers.get("Content-Type"),
        }
    except requests.RequestException as erro:
        return {
            "url": url,
            "status_code": None,
            "disponivel": False,
            "erro": str(erro),
        }

disponibilidade = pd.DataFrame([
    verificar_disponibilidade(montar_url_sia_pa(UF, ANO, mes))
    for mes in MESES
])

disponibilidade


## 6. Extração com PySUS

O `pysus` possui função simplificada para SIA/SUS:

```python
from pysus.api import sia
df = sia(state="SP", year=2024, month=[1, 2, 3], group="PA")
```


In [ ]:
def extrair_sia_pysus(uf: str, ano: int, meses: Iterable[int], grupo: str = "PA") -> pd.DataFrame:
    '''Extrai dados do SIA/SUS usando a API simplificada do PySUS.'''
    from pysus.api import sia

    uf = uf.upper().strip()
    meses = list(meses)

    try:
        df = sia(state=uf, year=ano, month=meses, group=grupo)
    except TypeError:
        df = sia(state=uf, year=ano, month=meses)

    if not isinstance(df, pd.DataFrame):
        try:
            df = df.to_pandas()
        except AttributeError:
            df = pd.DataFrame(df)

    return df


df_sia_raw = extrair_sia_pysus(UF, ANO, MESES, GRUPO)

print(f"Linhas extraídas: {df_sia_raw.shape[0]:,}")
print(f"Colunas extraídas: {df_sia_raw.shape[1]:,}")

df_sia_raw.head()


## 7. Quais dados serão extraídos

Os arquivos `PA` podem variar conforme competência/layout, mas os campos mais úteis para o projeto são:

| Campo provável | Uso no projeto |
|---|---|
| `PA_CMP` | Competência do atendimento |
| `PA_CODUNI` | Código CNES da unidade executante |
| `PA_UFMUN` | Município/UF do estabelecimento |
| `PA_MUNPCN` | Município de residência do paciente, quando disponível |
| `PA_PROC_ID` | Procedimento realizado; será enriquecido com SIGTAP |
| `PA_QTDPRO` | Quantidade produzida/apresentada |
| `PA_QTDAPR` | Quantidade aprovada |
| `PA_VALPRO` | Valor produzido/apresentado |
| `PA_VALAPR` | Valor aprovado |
| `PA_CBOCOD` | CBO/profissional associado ao procedimento |
| `PA_CIDPRI` | CID principal, quando aplicável |
| `PA_IDADE` | Idade do usuário |
| `PA_SEXO` | Sexo do usuário |
| `PA_RACACOR` | Raça/cor, quando preenchida |
| `PA_TPFIN` | Tipo de financiamento |
| `PA_SUBFIN` | Subtipo de financiamento |

A célula abaixo mostra as colunas efetivamente disponíveis no arquivo extraído.


In [ ]:
pd.DataFrame({
    "coluna": df_sia_raw.columns,
    "tipo_detectado": [str(df_sia_raw[col].dtype) for col in df_sia_raw.columns]
})


## 8. Seleção dos campos úteis para o MVP


In [ ]:
COLUNAS_INTERESSE = [
    "PA_CMP",
    "PA_CODUNI",
    "PA_UFMUN",
    "PA_MUNPCN",
    "PA_PROC_ID",
    "PA_QTDPRO",
    "PA_QTDAPR",
    "PA_VALPRO",
    "PA_VALAPR",
    "PA_CBOCOD",
    "PA_CIDPRI",
    "PA_IDADE",
    "PA_SEXO",
    "PA_RACACOR",
    "PA_TPFIN",
    "PA_SUBFIN",
]

colunas_existentes = [col for col in COLUNAS_INTERESSE if col in df_sia_raw.columns]
colunas_ausentes = [col for col in COLUNAS_INTERESSE if col not in df_sia_raw.columns]

df_sia = df_sia_raw[colunas_existentes].copy()

print("Colunas selecionadas:")
print(colunas_existentes)

print("\nColunas não encontradas neste recorte/layout:")
print(colunas_ausentes)

df_sia.head()


## 9. Tratamento básico dos tipos de dados


In [ ]:
def converter_colunas_sia(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "PA_CMP" in df.columns:
        # Competência costuma vir como AAAAMM.
        df["PA_CMP"] = df["PA_CMP"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(6)
        df["COMPETENCIA"] = pd.to_datetime(df["PA_CMP"] + "01", format="%Y%m%d", errors="coerce")

    campos_numericos = ["PA_QTDPRO", "PA_QTDAPR", "PA_VALPRO", "PA_VALAPR", "PA_IDADE"]
    for col in campos_numericos:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Padroniza códigos como texto
    for col in ["PA_CODUNI", "PA_UFMUN", "PA_MUNPCN", "PA_PROC_ID", "PA_CBOCOD", "PA_CIDPRI"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

    return df


df_sia = converter_colunas_sia(df_sia)
df_sia.info()


## 10. Dicionário analítico dos campos selecionados


In [ ]:
DICIONARIO_SIA = pd.DataFrame([
    {"campo": "COMPETENCIA", "descricao": "Competência mensal da produção ambulatorial."},
    {"campo": "PA_CODUNI", "descricao": "Código CNES da unidade executante."},
    {"campo": "PA_UFMUN", "descricao": "Município da unidade executante."},
    {"campo": "PA_MUNPCN", "descricao": "Município de residência do paciente, quando disponível."},
    {"campo": "PA_PROC_ID", "descricao": "Código do procedimento ambulatorial; deve ser enriquecido com SIGTAP."},
    {"campo": "PA_QTDPRO", "descricao": "Quantidade produzida/apresentada."},
    {"campo": "PA_QTDAPR", "descricao": "Quantidade aprovada."},
    {"campo": "PA_VALPRO", "descricao": "Valor produzido/apresentado."},
    {"campo": "PA_VALAPR", "descricao": "Valor aprovado."},
    {"campo": "PA_CBOCOD", "descricao": "CBO do profissional associado ao atendimento."},
    {"campo": "PA_CIDPRI", "descricao": "CID principal associado ao procedimento, quando aplicável."},
    {"campo": "PA_IDADE", "descricao": "Idade do usuário."},
    {"campo": "PA_SEXO", "descricao": "Sexo do usuário."},
])

DICIONARIO_SIA


## 11. Salvamento da base extraída


In [ ]:
arquivo_saida = DIRETORIO_TRATADO / f"sia_pa_{UF}_{ANO}_{'-'.join(str(m).zfill(2) for m in MESES)}.parquet"
df_sia.to_parquet(arquivo_saida, index=False)
print(f"Arquivo salvo em: {arquivo_saida}")


## 12. Perguntas que conseguimos responder com SIA/SUS

Com a base extraída, conseguimos responder perguntas ligadas à **demanda ambulatorial e acesso a especialidades**:

1. Quais procedimentos ambulatoriais têm maior volume?
2. Quais procedimentos especializados cresceram ao longo do tempo?
3. Quais municípios concentram maior produção ambulatorial?
4. Quais unidades executam mais consultas, exames ou procedimentos especializados?
5. Quais procedimentos têm maior diferença entre quantidade apresentada e aprovada?
6. Qual é o valor aprovado por procedimento, unidade ou município?
7. Quais CBOs/profissionais estão mais associados à produção ambulatorial?
8. Quais perfis de atendimento aparecem por idade e sexo?
9. Quais municípios têm baixa produção ambulatorial em relação à população?
10. Quais especialidades podem estar sob pressão, considerando produção SIA + oferta CNES?

Quando cruzado com SIGTAP, CNES e IBGE:

- **procedimentos por 10 mil habitantes**;
- **demanda por grupo/subgrupo de procedimento**;
- **ranking de especialidades mais pressionadas**;
- **demanda ambulatorial versus capacidade instalada**;
- **perfil de atendimento em crescimento por região**.


## 13. Exemplo 1 — Ranking de procedimentos ambulatoriais


In [ ]:
if "PA_PROC_ID" in df_sia.columns:
    agg = {}
    if "PA_QTDAPR" in df_sia.columns:
        agg["qtd_aprovada"] = ("PA_QTDAPR", "sum")
    if "PA_QTDPRO" in df_sia.columns:
        agg["qtd_produzida"] = ("PA_QTDPRO", "sum")
    if "PA_VALAPR" in df_sia.columns:
        agg["valor_aprovado"] = ("PA_VALAPR", "sum")

    ranking_procedimentos_sia = (
        df_sia.groupby("PA_PROC_ID")
        .agg(**agg)
        .reset_index()
        .sort_values(list(agg.keys())[0], ascending=False)
    )

    display(ranking_procedimentos_sia.head(20))
else:
    print("Campo PA_PROC_ID não encontrado.")


## 14. Exemplo 2 — Produção por estabelecimento CNES


In [ ]:
if "PA_CODUNI" in df_sia.columns:
    agg = {}
    if "PA_QTDAPR" in df_sia.columns:
        agg["qtd_aprovada"] = ("PA_QTDAPR", "sum")
    if "PA_VALAPR" in df_sia.columns:
        agg["valor_aprovado"] = ("PA_VALAPR", "sum")
    if "PA_PROC_ID" in df_sia.columns:
        agg["qtd_procedimentos_distintos"] = ("PA_PROC_ID", "nunique")

    producao_por_cnes = (
        df_sia.groupby("PA_CODUNI")
        .agg(**agg)
        .reset_index()
        .sort_values(list(agg.keys())[0], ascending=False)
    )

    display(producao_por_cnes.head(20))
else:
    print("Campo PA_CODUNI não encontrado.")


## 15. Exemplo 3 — Evolução mensal da produção ambulatorial


In [ ]:
if "COMPETENCIA" in df_sia.columns:
    agg = {}
    if "PA_QTDAPR" in df_sia.columns:
        agg["qtd_aprovada"] = ("PA_QTDAPR", "sum")
    if "PA_VALAPR" in df_sia.columns:
        agg["valor_aprovado"] = ("PA_VALAPR", "sum")

    evolucao_sia = (
        df_sia.groupby("COMPETENCIA")
        .agg(**agg)
        .reset_index()
        .sort_values("COMPETENCIA")
    )

    if "qtd_aprovada" in evolucao_sia.columns:
        evolucao_sia["crescimento_pct_qtd_aprovada"] = evolucao_sia["qtd_aprovada"].pct_change() * 100

    display(evolucao_sia)
else:
    print("Campo COMPETENCIA não criado.")


## 16. Como o SIA/SUS entra no Tema 4

O SIA/SUS não mede diretamente fila de espera, mas permite construir um proxy importante:

```text
Pressão ambulatorial por especialidade = volume de procedimentos especializados / oferta CNES
```

Com o IBGE:

```text
Procedimentos especializados por 10 mil habitantes = (qtd aprovada SIA / população IBGE) * 10.000
```

Com o SIGTAP:

```text
Procedimento SIA → grupo, subgrupo, forma de organização, nome e complexidade
```

Esse enriquecimento permite transformar códigos técnicos em uma leitura executiva para gestores.
